# Homework Starter — Stage 15: Orchestration & System Design
Complete the sections below. Keep your answers concise and focused on orchestration readiness.

## 1) Project Task Decomposition
List 4–8 tasks. Add more rows as needed.

In [ ]:
from pathlib import Path
import pandas as pd
tasks = pd.DataFrame({
    'task': ['ingest', 'clean', 'train_or_score', 'report'],
    'inputs': ['/data/raw.ext', 'prices_raw.json', 'prices_clean.json', 'model.json'],
    'outputs': ['prices_raw.json', 'prices_clean.json', 'model.json', 'report.txt'],
    'idempotent': [True, True, True, True]
})
tasks

In [1]:
from pathlib import Path
import pandas as pd

tasks = pd.DataFrame({
    'task': ['ingest_data', 'validate_data', 'feature_engineering', 'train_model', 'generate_predictions', 'backtest_strategy', 'generate_report', 'deploy_monitor'],
    'inputs': ['yfinance API', 'raw_data.parquet', 'validated_data.parquet', 'features.parquet', 'model.pkl + features.parquet', 'predictions.parquet', 'backtest_results.parquet', 'report.html + model.pkl'],
    'outputs': ['raw_data.parquet', 'validated_data.parquet', 'features.parquet', 'model.pkl + metrics.json', 'predictions.parquet', 'backtest_results.parquet', 'report.html', 'monitoring_dashboard.json'],
    'idempotent': [False, True, True, True, True, True, True, True]
})
tasks

,task,inputs,outputs,idempotent
0,ingest_data,yfinance API,raw_data.parquet,False
1,validate_data,raw_data.parquet,validated_data.parquet,True
2,feature_engineering,validated_data.parquet,features.parquet,True
3,train_model,features.parquet,model.pkl + metrics.json,True
4,generate_predictions,model.pkl + features.parquet,predictions.parquet,True
5,backtest_strategy,predictions.parquet,backtest_results.parquet,True
6,generate_report,backtest_results.parquet,report.html,True
7,deploy_monitor,report.html + model.pkl,monitoring_dashboard.json,True


## 2) Dependencies (DAG)
Describe dependencies and paste a small diagram if you have one.

In [ ]:
dag = {
    'ingest': [],
    'clean': ['ingest'],
    'train_or_score': ['clean'],
    'report': ['train_or_score']
}
dag

In [2]:
dag = {
    'ingest_data': [],
    'validate_data': ['ingest_data'],
    'feature_engineering': ['validate_data'],
    'train_model': ['feature_engineering'],
    'generate_predictions': ['train_model', 'feature_engineering'],
    'backtest_strategy': ['generate_predictions'],
    'generate_report': ['backtest_strategy', 'train_model'],
    'deploy_monitor': ['generate_predictions', 'train_model']
}

# Visual DAG representation:
# ingest_data → validate_data → feature_engineering → train_model → generate_predictions → backtest_strategy → generate_report
#                                                              ↘ generate_predictions → deploy_monitor
# Note: train_model and generate_predictions can run in parallel after feature_engineering

## 3) Logging & Checkpoints Plan
Specify what you will log and where you will checkpoint for each task.

In [ ]:
logging_plan = pd.DataFrame({
    'task': ['ingest', 'clean', 'train_or_score', 'report'],
    'log_messages': [
        'start/end, rows, source URI',
        'start/end, rows in/out',
        'params, metrics',
        'artifact path'
    ],
    'checkpoint_artifact': [
        'prices_raw.json',
        'prices_clean.json',
        'model.json',
        'report.txt'
    ]
})
logging_plan

In [3]:
logging_plan = pd.DataFrame({
    'task': ['ingest_data', 'validate_data', 'feature_engineering', 'train_model', 'generate_predictions', 'backtest_strategy', 'generate_report', 'deploy_monitor'],
    'log_messages': [
        'start/end, rows ingested, API status, timestamp',
        'start/end, validation errors, row counts, schema changes',
        'start/end, features created, null counts, transformation stats',
        'start/end, model params, training metrics, feature importance',
        'start/end, prediction stats, confidence intervals',
        'start/end, strategy performance, Sharpe ratio, max drawdown',
        'start/end, report generated, visualization status',
        'start/end, monitoring metrics, drift detection, alert status'
    ],
    'checkpoint_artifact': [
        'data/raw/raw_data_{date}.parquet',
        'data/processed/validated_data_{date}.parquet',
        'data/features/features_{date}.parquet',
        'models/model_{date}.pkl + metrics/metrics_{date}.json',
        'predictions/predictions_{date}.parquet',
        'backtests/backtest_{date}.parquet',
        'reports/report_{date}.html',
        'monitoring/dashboard_{date}.json'
    ]
})
logging_plan

,task,log_messages,checkpoint_artifact
0,ingest_data,"start/end, rows ingested, API status, timestamp",data/raw/raw_data_{date}.parquet
1,validate_data,"start/end, validation errors, row counts, sche...",data/processed/validated_data_{date}.parquet
2,feature_engineering,"start/end, features created, null counts, tran...",data/features/features_{date}.parquet
3,train_model,"start/end, model params, training metrics, fea...",models/model_{date}.pkl + metrics/metrics_{dat...
4,generate_predictions,"start/end, prediction stats, confidence intervals",predictions/predictions_{date}.parquet
5,backtest_strategy,"start/end, strategy performance, Sharpe ratio,...",backtests/backtest_{date}.parquet
6,generate_report,"start/end, report generated, visualization status",reports/report_{date}.html
7,deploy_monitor,"start/end, monitoring metrics, drift detection...",monitoring/dashboard_{date}.json


## 4) Right-Sizing Automation
Which parts will you automate now? Which stay manual? Why?

Automate Now:
- Data ingestion and validation (daily automated pull from yfinance API)
- Feature engineering pipeline (consistent transformation logic)
- Model training and prediction generation
- Basic reporting and monitoring

Keep Manual:
- Model selection and hyperparameter tuning (quarterly review)
- Strategy parameter optimization (monthly review)
- Deployment approval to production (human-in-the-loop validation)
- Major schema changes and feature additions

Rationale: Automate repetitive, well-defined tasks to ensure consistency and reliability. Keep human oversight for strategic decisions, model validation, and production deployments to mitigate risks in financial applications.

## 5) (Stretch) Refactor One Task into a Function + CLI
Use the templates below.

In [ ]:
import argparse, json, logging, sys
from datetime import datetime

def my_task(input_path: str, output_path: str) -> None:
    '''Example task template: read → transform → write JSON.'''
    logging.info('[my_task] start')
    # TODO: implement your logic
    result = {'run_at': datetime.utcnow().isoformat(), 'note': 'replace with real output'}
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    Path(output_path).write_text(json.dumps(result, indent=2))
    logging.info('[my_task] wrote %s', output_path)

def main(argv=None):
    parser = argparse.ArgumentParser(description='Homework task wrapper')
    parser.add_argument('--input', required=True)
    parser.add_argument('--output', required=True)
    args = parser.parse_args(argv)
    logging.basicConfig(level=logging.INFO, handlers=[logging.StreamHandler(sys.stdout)])
    my_task(args.input, args.output)

if **name** == '**main**'':
    # Example simulated CLI in notebook:
    main(['--input', 'data/in.ext', '--output', 'data/out.json'])

In [4]:
import argparse
import json
import logging
import sys
from datetime import datetime
from pathlib import Path
import pandas as pd
import yfinance as yf

def ingest_data(output_path: str, ticker: str = "SPY", period: str = "2y") -> None:
    '''Ingest stock data from yfinance API and save to parquet'''
    logging.info('[ingest_data] start for ticker %s', ticker)
    
    try:
        # Fetch data
        data = yf.download(ticker, period=period, progress=False)
        
        # Add metadata
        data['ingestion_timestamp'] = datetime.utcnow()
        data['ticker'] = ticker
        
        # Ensure output directory exists
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        # Save to parquet
        data.to_parquet(output_path)
        logging.info('[ingest_data] wrote %s with %d rows', output_path, len(data))
        
    except Exception as e:
        logging.error('[ingest_data] failed: %s', str(e))
        raise

def main(argv=None):
    parser = argparse.ArgumentParser(description='Data ingestion task for SPY ETF')
    parser.add_argument('--output', required=True, help='Output parquet file path')
    parser.add_argument('--ticker', default='SPY', help='Stock ticker symbol')
    parser.add_argument('--period', default='2y', help='Time period to download')
    
    args = parser.parse_args(argv)
    logging.basicConfig(level=logging.INFO, 
                       format='%(asctime)s - %(levelname)s - %(message)s',
                       handlers=[logging.StreamHandler(sys.stdout)])
    
    ingest_data(args.output, args.ticker, args.period)

if __name__ == '__main__':
    # Example CLI call within notebook for testing
    main(['--output', 'data/raw/spy_data.parquet', '--ticker', 'SPY', '--period', '2y'])

2025-08-27 13:16:33,085 - INFO - [ingest_data] start for ticker SPY


/var/folders/_4/t03mdfy94ts0ylt8cw0q1bp00000gn/T/ipykernel_32667/832384102.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, progress=False)


2025-08-27 13:16:33,828 - INFO - [ingest_data] wrote data/raw/spy_data.parquet with 502 rows


### Optional: Simple Retry Wrapper (fill in)
Add a small retry with linear backoff to harden a task.

In [ ]:
import time
def retry(n_tries=3, delay=0.2):
    def wrapper(fn, *args, **kwargs):
        # TODO: implement try/except loop with sleep backoff
        return fn(*args, **kwargs)
    return wrapper

In [5]:
import time
from typing import Callable

def retry(n_tries: int = 3, delay: float = 1.0, backoff: float = 2.0):
    """Retry decorator with exponential backoff"""
    def decorator(func: Callable):
        def wrapper(*args, **kwargs):
            attempts = 0
            current_delay = delay
            
            while attempts < n_tries:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    if attempts == n_tries:
                        logging.error('[retry] Failed after %d attempts: %s', n_tries, str(e))
                        raise
                    
                    logging.warning('[retry] Attempt %d failed: %s. Retrying in %.1fs', 
                                  attempts, str(e), current_delay)
                    time.sleep(current_delay)
                    current_delay *= backoff  # Exponential backoff
        return wrapper
    return decorator

# Example usage with retry
@retry(n_tries=3, delay=2.0, backoff=2.0)
def robust_ingest_data(output_path: str, ticker: str = "SPY", period: str = "2y") -> None:
    '''Ingest data with retry logic for transient failures'''
    ingest_data(output_path, ticker, period)